In [1]:
import pandas as pd
import re
import json

In [ ]:
df = pd.read_csv("KC_PET_ACP_CTLSTT_LC_DATA_2023.csv")

In [3]:
import re
import pymysql
import pandas as pd

# pet_size 파싱 함수 그대로
def parse_pet_size(text):
    if text is None or str(text).strip() == "":
        return [0]

    text = str(text)

    if "모두" in text:
        return [4]

    sizes = []

    # 몸무게 기반
    weights = list(map(int, re.findall(r"\d+", text)))
    for w in weights:
        if w <= 10:
            sizes.append(1)
        elif 10 < w <= 25:
            sizes.append(2)
        else:
            sizes.append(3)

    # 키워드 기반
    if "소형" in text:
        sizes.append(1)
    if "중형" in text:
        sizes.append(2)
    if "대형" in text:
        sizes.append(3)

    return list(set(sizes)) if sizes else [0]

# DB 연결
conn = pymysql.connect(
    host="localhost",
    user="mini",
    password="mini",
    db="miniproject",
    charset="utf8mb4"
)
cursor = conn.cursor()

# place 테이블에서 id, fclty_nm, ADDR 가져오기
cursor.execute("SELECT id, fclty_nm, LNM_ADDR FROM place")
place_map = {
    (str(name).strip(), str(addr).strip()): pid
    for pid, name, addr in cursor.fetchall()
}

# insert SQL
sql = """
INSERT INTO sizefilterdata (
    place_id,
    pet_size,
    ENTRN_POSBL_PET_SIZE_VALUE
) VALUES (%s, %s, %s)
"""

# df iterate
for _, r in df.iterrows():
    place_name = str(r["FCLTY_NM"]).strip()
    place_addr = str(r["LNM_ADDR"]).strip()  # 주소 컬럼
    oper_text = r["ENTRN_POSBL_PET_SIZE_VALUE"]

    place_id = place_map.get((place_name, place_addr))
    if place_id is None:
        print(f"❗ place_id 매핑 실패: {place_name} / {place_addr}")
        continue

    pet_sizes = parse_pet_size(oper_text)

    for pet_size in pet_sizes:
        cursor.execute(sql, (place_id, pet_size, oper_text))

conn.commit()
conn.close()
